In [1]:
import pandas as pd

In [3]:
data=pd.read_csv('cyber_security_logs.xls')
data.head()

,Attack1,Attack2,Attack3,Attack4
0,Brute Force,Malware,Phishing,NaN
1,Brute Force,Malware,Ransomware,NaN
2,Phishing,Credential Theft,NaN,NaN
3,Brute Force,Malware,Credential Theft,NaN
4,Phishing,Ransomware,NaN,NaN


In [8]:
transactions = []

for _, row in data.iterrows():
    
    transaction = []

    for item in row:
        if pd.notna(item):
            transaction.append(str(item))

    transactions.append(transaction)

print(transactions[:5])


[['Brute Force', 'Malware', 'Phishing'], ['Brute Force', 'Malware', 'Ransomware'], ['Phishing', 'Credential Theft'], ['Brute Force', 'Malware', 'Credential Theft'], ['Phishing', 'Ransomware']]


In [9]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

encoded_df = pd.DataFrame(
    te_array,
    columns=te.columns_
)

print(encoded_df.head())

   Brute Force  Credential Theft  Malware  Phishing  Ransomware
0         True             False     True      True       False
1         True             False     True     False        True
2        False              True    False      True       False
3         True              True     True     False       False
4        False             False    False      True        True


In [10]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(
    encoded_df,
    min_support=0.2,
    use_colnames=True
)

print(frequent_itemsets)

     support                         itemsets
0   0.489796                    (Brute Force)
1   0.551020               (Credential Theft)
2   0.775510                        (Malware)
3   0.469388                       (Phishing)
4   0.489796                     (Ransomware)
5   0.224490  (Credential Theft, Brute Force)
6   0.387755           (Brute Force, Malware)
7   0.204082        (Brute Force, Ransomware)
8   0.367347      (Credential Theft, Malware)
9   0.204082     (Credential Theft, Phishing)
10  0.204082   (Credential Theft, Ransomware)
11  0.367347              (Phishing, Malware)
12  0.367347            (Ransomware, Malware)


In [11]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
)

print(rules)

          antecedents    consequents  antecedent support  consequent support  \
0       (Brute Force)      (Malware)            0.489796            0.775510   
1           (Malware)  (Brute Force)            0.775510            0.489796   
2  (Credential Theft)      (Malware)            0.551020            0.775510   
3          (Phishing)      (Malware)            0.469388            0.775510   
4        (Ransomware)      (Malware)            0.489796            0.775510   

    support  confidence      lift  representativity  leverage  conviction  \
0  0.387755    0.791667  1.020833               1.0  0.007913    1.077551   
1  0.387755    0.500000  1.020833               1.0  0.007913    1.020408   
2  0.367347    0.666667  0.859649               1.0 -0.059975    0.673469   
3  0.367347    0.782609  1.009153               1.0  0.003332    1.032653   
4  0.367347    0.750000  0.967105               1.0 -0.012495    0.897959   

   zhangs_metric   jaccard  certainty  kulczynski  
0   

In [12]:
print(
    rules[
        [
            'antecedents',
            'consequents',
            'support',
            'confidence',
            'lift'
        ]
    ]
)

          antecedents    consequents   support  confidence      lift
0       (Brute Force)      (Malware)  0.387755    0.791667  1.020833
1           (Malware)  (Brute Force)  0.387755    0.500000  1.020833
2  (Credential Theft)      (Malware)  0.367347    0.666667  0.859649
3          (Phishing)      (Malware)  0.367347    0.782609  1.009153
4        (Ransomware)      (Malware)  0.367347    0.750000  0.967105


In [13]:
from mlxtend.frequent_patterns import fpgrowth

frequent_itemsets = fpgrowth(
    encoded_df,
    min_support=0.2,
    use_colnames=True
)

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.5
)

print(rules[
    [
        'antecedents',
        'consequents',
        'support',
        'confidence',
        'lift'
    ]
])

          antecedents    consequents   support  confidence      lift
0       (Brute Force)      (Malware)  0.387755    0.791667  1.020833
1           (Malware)  (Brute Force)  0.387755    0.500000  1.020833
2          (Phishing)      (Malware)  0.367347    0.782609  1.009153
3        (Ransomware)      (Malware)  0.367347    0.750000  0.967105
4  (Credential Theft)      (Malware)  0.367347    0.666667  0.859649


In [14]:
for _, row in rules.iterrows():

    antecedent = list(row['antecedents'])
    consequent = list(row['consequents'])

    print(
        f"{antecedent} ---> {consequent}"
    )

['Brute Force'] ---> ['Malware']
['Malware'] ---> ['Brute Force']
['Phishing'] ---> ['Malware']
['Ransomware'] ---> ['Malware']
['Credential Theft'] ---> ['Malware']
